# GeneFlow AI · Entrenamiento completo en Kaggle

Este notebook entrena **sin supervisión** la mejor configuración de la búsqueda de hiperparámetros con el conjunto de entrenamiento completo, aprovechando una sesión de Kaggle de 12 horas con **dos GPU T4**.

## Cómo lanzarlo

1. En Kaggle: **Create → New Notebook → File → Import Notebook** y sube este archivo.
2. En **Session options**: **Accelerator: GPU T4 ×2** e **Internet: On**.
3. **Save Version → Save & Run All (Commit)**. Puedes cerrar el navegador.
4. Cuando termine, descarga la carpeta **`train`** de la pestaña **Output**. La versión guardada conserva también todas las salidas de las celdas: el progreso de la demo, los informes periódicos y las gráficas.

## Qué hace

| Paso | Qué ocurre | Tiempo aproximado |
|---|---|---|
| 1 | Entorno: GPU, PyTorch con la versión de CUDA compatible, repositorio y dependencias | 5-8 min |
| 2 | Datos: `geneflow prepare` y `geneflow cache` | 12-20 min |
| 3 | **Demo**: un entrenamiento corto con el progreso en directo y su reporte, a modo de vista previa | 3-5 min |
| 4 | Plan: cuántas épocas caben en el tiempo que queda | instantáneo |
| 5 | **Entrenamiento completo** en las dos GPU en paralelo, con un informe cada 10 minutos | el resto de la sesión |
| 6 | **Reportes**: curvas de pérdida, accuracy por nivel, comparación entre ejecuciones y tabla final | 1 min |

## Qué entrena

| GPU | Configuración | Origen | Velocidad en T4 | Época completa |
|---|---|---|---|---|
| 0 | **CNN, prueba 12**: `base`, dos bloques, kernel 5, lr 8,2e-4, dropout 0,05 | `reports/search/cnn-best.json`, la mejor de la búsqueda (0,605) | ~405 secuencias/s | ~59 min |
| 1 | CNN, prueba 12 con **otra semilla** | la misma configuración | ~405 secuencias/s | ~59 min |

Cada época recorre **1,4 millones de secuencias** de train, reales y sintéticas, muestreadas con el peso por frecuencia de la búsqueda. La validación usa las **62 mil secuencias** de `val` completas. `test` no se toca.

La segunda GPU entrena la misma configuración con otra semilla. Así se mide la **variabilidad entre ejecuciones**: si las dos acaban muy cerca, las diferencias entre modelos que se vean después son de fiar. Si prefieres aprovecharla para el finalista grande (prueba 20, `large`, 0,597 en la búsqueda), cambia la segunda entrada de `RUNS` por la que está comentada.

## La demo

Antes de gastar la sesión, la GPU 0 entrena la misma configuración durante **2 épocas de 12.800 secuencias**, mostrando en la celda cada 20 lotes la pérdida, la velocidad y el learning rate, y al final de cada época la validación. Después se genera su reporte con las mismas gráficas que tendrá el entrenamiento completo. Sirve para ver cómo se comporta el entrenamiento y para comprobar que todo funciona: si la demo falla, el notebook se detiene ahí.

## Presupuesto

Las épocas se calculan con el tiempo que queda después de la demo: el 92 % del tiempo restante dividido entre la duración estimada de una época, con un máximo de `MAX_EPOCHS`. Con la sesión completa salen unas **10 épocas**, unos **14 millones de secuencias vistas**, 23 veces más que en la búsqueda. El calendario del learning rate (calentamiento y coseno) se ajusta a ese número de épocas.

**Seguridad.** Cada entrenamiento recibe además como límite estricto el tiempo que queda de sesión menos 30 minutos. Si se acerca, se corta en el lote en curso y conserva los checkpoints de las épocas ya terminadas. Así el notebook siempre acaba a tiempo de guardar las salidas.

## Qué se guarda en `train/`

| Archivo | Contenido |
|---|---|
| `<nombre>/best.pt` | Los pesos de la época con menor pérdida de validación |
| `<nombre>/last.pt` | Los pesos de la última época |
| `<nombre>/history.json` | Pérdida y accuracy por nivel en cada época |
| `<nombre>/run.json` | La configuración completa: hiperparámetros, encoder, cabezas, entrenamiento y número de parámetros |
| `<nombre>/label_space.json` | El espacio de etiquetas con el que se entrenó, necesario para evaluar el modelo |
| `<nombre>/train.log` | El registro completo del entrenamiento |
| `reports/demo-training.png`, `reports/final-training.png` | Las gráficas de la demo y del entrenamiento completo |
| `reports/demo-summary.json`, `reports/final-summary.json` | La mejor época de cada ejecución con su accuracy por nivel |


In [ ]:
import json
import math
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

SESSION_START = time.time()
SESSION_HOURS = 12.0
SAFETY_MARGIN_HOURS = 0.5
EPOCH_BUDGET_FRACTION = 0.92
MAX_EPOCHS = 12

REPOSITORY = "https://github.com/Dexaroz/geneflow-ai-taxonomy-classifier.git"
BRANCH = "main"

WORK = Path("/tmp/geneflow")
REPO = WORK / "repo"
DATA = WORK / "data"
OUTPUT = Path("/kaggle/working/train")
REPORTS = OUTPUT / "reports"
GENEFLOW = REPO / ".venv" / "bin" / "geneflow"
PYTHON = REPO / ".venv" / "bin" / "python"

RUNS = [
    {"name": "cnn-trial12", "params": "reports/search/cnn-best.json", "trial": 12, "seed": 20260923, "sequences_per_second": 405},
    {"name": "cnn-trial12-seed2", "params": "reports/search/cnn-best.json", "trial": 12, "seed": 20260924, "sequences_per_second": 405},
    # {"name": "cnn-trial20", "params": "reports/search/cnn-trials.json", "trial": 20, "seed": 20260923, "sequences_per_second": 301},
]

TRAIN_ARGUMENTS = [
    "--batch-size", "128",
    "--precision", "fp16",
    "--gpu-memory-fraction", "0.92",
]

DEMO = {"name": "demo", "epochs": 2, "samples_per_epoch": 12_800, "log_every": 20}
DEMO_DIR = WORK / "demo"

LEVELS = ["domain", "kingdom", "phylum", "class", "order", "family", "genus", "species"]
OBJECTIVE_LEVELS = ["genus", "species"]

EVALUATION_SPEEDUP = 3.0
REPORT_EVERY_SECONDS = 600

WORK.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)


def run(command, **kwargs):
    print("$", " ".join(str(part) for part in command), flush=True)

    return subprocess.run([str(part) for part in command], check=True, **kwargs)


def stream(command, log_path, environment):
    print("$", " ".join(str(part) for part in command), flush=True)

    with open(log_path, "w") as log:
        process = subprocess.Popen(
            [str(part) for part in command],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=environment,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            log.write(line)
            print(line.rstrip()[-200:], flush=True)

    if process.wait() != 0:
        raise RuntimeError(f"{command[1]} terminó con código {process.returncode}; revisa {log_path}")


def elapsed_hours():
    return (time.time() - SESSION_START) / 3600


def remaining_hours():
    return SESSION_HOURS - SAFETY_MARGIN_HOURS - elapsed_hours()


print(f"Python del notebook: {sys.version.split()[0]} | trabajo en {WORK} | salidas en {OUTPUT}")

## 1. Entorno: GPU, PyTorch, repositorio y dependencias

In [ ]:
gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=True,
)
gpus = [line.split(", ") for line in gpu_query.stdout.strip().splitlines()]
driver_major = int(gpus[0][1].split(".")[0])
cuda_index = "cu130" if driver_major >= 580 else "cu126"

for index, (name, driver, memory) in enumerate(gpus):
    print(f"GPU {index}: {name} | driver {driver} | {memory}")

print(f"PyTorch se instalará con {cuda_index}")

run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])

if REPO.exists():
    shutil.rmtree(REPO)

run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%h %s"])

run(["uv", "python", "install", "3.14"])
run(["uv", "sync", "--frozen", "--no-default-groups", "--no-install-package", "torch"], cwd=REPO)
run(["uv", "pip", "install", "--python", PYTHON, "torch==2.14.0", "--index-url", f"https://download.pytorch.org/whl/{cuda_index}"], cwd=REPO)

run([PYTHON, "-c", "import torch; print('torch', torch.__version__, '| GPU visibles:', torch.cuda.device_count(), '|', torch.cuda.get_device_name(0))"])

print(f"Entorno listo en {elapsed_hours() * 60:.0f} min")

## 2. Datos: descarga, construcción, datos sintéticos y cachés

In [ ]:
run([GENEFLOW, "prepare", "--data-dir", DATA])
run([GENEFLOW, "cache", "--data-dir", DATA])

train_sequences = json.loads((DATA / "interim" / "geneflow" / "train_tokens.json").read_text())["sequences"]
validation_sequences = json.loads((DATA / "interim" / "geneflow" / "val_tokens.json").read_text())["sequences"]

print(f"Train: {train_sequences:,} secuencias | val: {validation_sequences:,}")
print(f"Datos listos a los {elapsed_hours() * 60:.0f} min de sesión")

## Funciones de reporte

El mismo reporte sirve para la demo y para el entrenamiento completo:

- **Pérdida** de entrenamiento (continua) y de validación (discontinua) en cada época.
- **Objetivo de la búsqueda**, la media de la accuracy de género y especie, frente a la mejor prueba de la búsqueda (línea gris), que se entrenó 6 épocas de 100 mil secuencias.
- **Accuracy por nivel** en cada época, de dominio a especie.
- **Accuracy por nivel en la mejor época** (la de menor pérdida de validación) de cada ejecución, lado a lado.

Las figuras y un resumen en JSON se guardan en `train/reports/`.

In [ ]:
import matplotlib.pyplot as plt

SEARCH_BEST = json.loads((REPO / "reports" / "search" / "cnn-best.json").read_text())["value"]


def objective(record):
    return sum(record["val_accuracy"][level] for level in OBJECTIVE_LEVELS) / len(OBJECTIVE_LEVELS)


def load_histories(directories):
    histories = {}

    for name, directory in directories.items():
        path = Path(directory) / "history.json"

        if path.exists():
            histories[name] = json.loads(path.read_text())

    return histories


def training_report(directories, prefix):
    histories = load_histories(directories)

    if not histories:
        print("Sin épocas terminadas: no hay nada que reportar")

        return None

    figure, axes = plt.subplots(2, 2, figsize=(15, 10))
    loss_axis, objective_axis, level_axis, best_axis = axes.flat
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    level_colors = plt.cm.viridis([index / (len(LEVELS) - 1) for index in range(len(LEVELS))])

    for (name, history), color in zip(histories.items(), colors):
        epochs = [record["epoch"] for record in history]

        loss_axis.plot(epochs, [record["train_loss"] for record in history], "o-", color=color, label=f"{name} · train")
        loss_axis.plot(epochs, [record["val_loss"] for record in history], "o--", color=color, label=f"{name} · val")
        objective_axis.plot(epochs, [objective(record) for record in history], "o-", color=color, label=name)

    objective_axis.axhline(SEARCH_BEST, color="grey", linestyle=":", label=f"mejor de la búsqueda ({SEARCH_BEST:.3f})")

    first_name, first_history = next(iter(histories.items()))
    first_epochs = [record["epoch"] for record in first_history]

    for level, color in zip(LEVELS, level_colors):
        level_axis.plot(first_epochs, [record["val_accuracy"][level] for record in first_history], "o-", color=color, label=level)

    bests = {name: min(history, key=lambda record: record["val_loss"]) for name, history in histories.items()}
    width = 0.8 / len(bests)

    for offset, ((name, best), color) in enumerate(zip(bests.items(), colors)):
        positions = [index + offset * width - 0.4 + width / 2 for index in range(len(LEVELS))]
        best_axis.bar(positions, [best["val_accuracy"][level] for level in LEVELS], width, color=color, label=f"{name} (época {best['epoch']})")

    loss_axis.set(title="Pérdida", xlabel="época", ylabel="pérdida jerárquica")
    objective_axis.set(title="Objetivo: media de género y especie", xlabel="época", ylabel="accuracy", ylim=(0, 1))
    level_axis.set(title=f"Accuracy por nivel · {first_name}", xlabel="época", ylabel="accuracy", ylim=(0, 1))
    best_axis.set(title="Accuracy por nivel en la mejor época", ylabel="accuracy", ylim=(0, 1))
    best_axis.set_xticks(range(len(LEVELS)), LEVELS, rotation=30)

    for axis in (loss_axis, objective_axis, level_axis, best_axis):
        axis.grid(alpha=0.3)
        axis.legend(fontsize=8)

    for axis in (loss_axis, objective_axis, level_axis):
        axis.xaxis.get_major_locator().set_params(integer=True)

    figure.suptitle(f"GeneFlow · {prefix}", fontsize=14)
    figure.tight_layout()
    figure.savefig(REPORTS / f"{prefix}-training.png", dpi=150)
    plt.show()

    summary = {
        name: {
            "best_epoch": best["epoch"],
            "epochs": len(histories[name]),
            "val_loss": best["val_loss"],
            "objective": objective(best),
            "val_accuracy": best["val_accuracy"],
            "minutes": sum(record["seconds"] for record in histories[name]) / 60,
        }
        for name, best in bests.items()
    }
    (REPORTS / f"{prefix}-summary.json").write_text(json.dumps(summary, indent=2))

    header = f"{'ejecución':<22} {'época':>5} {'val loss':>9} {'objetivo':>9} " + " ".join(f"{level[:7]:>7}" for level in LEVELS) + f" {'min':>6}"
    print(header)
    print("-" * len(header))

    for name, row in summary.items():
        print(
            f"{name:<22} {row['best_epoch']:>2}/{row['epochs']:<2} {row['val_loss']:>9.4f} {row['objective']:>9.4f} "
            + " ".join(f"{row['val_accuracy'][level]:>7.3f}" for level in LEVELS)
            + f" {row['minutes']:>6.0f}"
        )

    print(f"\nReferencia: mejor prueba de la búsqueda {SEARCH_BEST:.4f} (6 épocas × 100 mil secuencias, val de 20 mil)")

    return summary

## 3. Demo: un entrenamiento corto en directo

La misma configuración que el entrenamiento completo, durante 2 épocas de 12.800 secuencias. Cada 20 lotes se muestran la pérdida media, la velocidad y el learning rate; al final de cada época, la validación completa. Con tan pocas secuencias la accuracy será baja: lo que importa es ver que la pérdida baja y que el reporte se genera.

In [ ]:
demo_run = RUNS[0]

if DEMO_DIR.exists():
    shutil.rmtree(DEMO_DIR)

DEMO_DIR.mkdir(parents=True)

stream(
    [
        GENEFLOW, "train",
        "--data-dir", DATA,
        "--params", REPO / demo_run["params"],
        "--trial", str(demo_run["trial"]),
        "--output", DEMO_DIR,
        "--epochs", str(DEMO["epochs"]),
        "--samples-per-epoch", str(DEMO["samples_per_epoch"]),
        "--log-every", str(DEMO["log_every"]),
        "--seed", str(demo_run["seed"]),
        *TRAIN_ARGUMENTS,
    ],
    DEMO_DIR / "train.log",
    dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1"),
)

run_description = json.loads((DEMO_DIR / "run.json").read_text())
print(f"\nModelo: {run_description['parameters']['total']:,} parámetros ({run_description['parameters']['encoder']:,} en el encoder)")

demo_summary = training_report({"demo": DEMO_DIR}, "demo")
print(f"\nDemo terminada a los {elapsed_hours() * 60:.0f} min de sesión")

## 4. Plan: épocas que caben en el tiempo restante

In [ ]:
active_runs = RUNS[: len(gpus)]


def epoch_minutes(sequences_per_second):
    training = train_sequences / sequences_per_second
    evaluation = validation_sequences / (sequences_per_second * EVALUATION_SPEEDUP)

    return (training + evaluation) / 60


for entry in active_runs:
    minutes = epoch_minutes(entry["sequences_per_second"])
    entry["epochs"] = max(1, min(MAX_EPOCHS, math.floor(remaining_hours() * 60 * EPOCH_BUDGET_FRACTION / minutes)))

    print(
        f"{entry['name']}: {entry['epochs']} épocas de ~{minutes:.0f} min "
        f"(~{entry['epochs'] * minutes / 60:.1f} h, {entry['epochs'] * train_sequences / 1e6:.1f} M secuencias)"
    )

print(f"Tiempo disponible: {remaining_hours():.2f} h")

## 5. Entrenamiento completo en paralelo, con un informe cada 10 minutos

In [ ]:
def launch(entry, gpu):
    directory = OUTPUT / entry["name"]
    directory.mkdir(parents=True, exist_ok=True)

    environment = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    log = open(directory / "train.log", "a")

    command = [
        GENEFLOW, "train",
        "--data-dir", DATA,
        "--params", REPO / entry["params"],
        "--trial", str(entry["trial"]),
        "--output", directory,
        "--epochs", str(entry["epochs"]),
        "--seed", str(entry["seed"]),
        "--hours", f"{remaining_hours():.3f}",
        *TRAIN_ARGUMENTS,
    ]

    print(f"Lanzando {entry['name']} en la GPU {gpu}: {entry['epochs']} épocas")

    return subprocess.Popen([str(part) for part in command], stdout=log, stderr=subprocess.STDOUT, env=environment)


def progress(entry):
    history_path = OUTPUT / entry["name"] / "history.json"

    if not history_path.exists():
        return "sin épocas terminadas"

    history = json.loads(history_path.read_text())
    last = history[-1]
    accuracy = last["val_accuracy"]

    return (
        f"época {last['epoch']}/{entry['epochs']} | val loss {last['val_loss']:.4f} | "
        f"género {accuracy['genus']:.3f} | especie {accuracy['species']:.3f} | objetivo {objective(last):.4f}"
    )


def last_line(entry):
    path = OUTPUT / entry["name"] / "train.log"
    lines = path.read_text(errors="replace").strip().splitlines() if path.exists() else []

    return lines[-1][-160:] if lines else "(sin salida todavía)"


processes = {entry["name"]: launch(entry, gpu) for gpu, entry in enumerate(active_runs)}

while any(process.poll() is None for process in processes.values()):
    time.sleep(REPORT_EVERY_SECONDS)

    print(f"\n[{elapsed_hours():.2f} h de sesión, quedan {max(0.0, remaining_hours()):.2f} h]")

    for entry in active_runs:
        print(f"  {entry['name']}: {progress(entry)}")
        print(f"    {last_line(entry)}")

for name, process in processes.items():
    print(f"{name} terminó con código {process.returncode}")

print(f"\nEntrenamiento terminado a las {elapsed_hours():.2f} h de sesión")

## 6. Reportes del entrenamiento completo

Las gráficas y la tabla de la mejor época de cada ejecución. Con dos semillas, la diferencia entre ambas da una idea del ruido entre ejecuciones de la misma configuración.

In [ ]:
final_summary = training_report({entry["name"]: OUTPUT / entry["name"] for entry in active_runs}, "final")

if final_summary and len(final_summary) > 1:
    objectives = [row["objective"] for row in final_summary.values()]
    print(f"\nDiferencia de objetivo entre ejecuciones: {max(objectives) - min(objectives):.4f}")

for entry in active_runs:
    history_path = OUTPUT / entry["name"] / "history.json"

    if not history_path.exists():
        continue

    print(f"\n=== {entry['name']} época a época")

    for record in json.loads(history_path.read_text()):
        accuracy = record["val_accuracy"]
        print(
            f"  época {record['epoch']:>2}: train {record['train_loss']:.4f} | val {record['val_loss']:.4f} | "
            + " ".join(f"{level[:4]} {accuracy[level]:.3f}" for level in LEVELS)
            + f" | {record['seconds'] / 60:.0f} min"
        )

print("\nArchivos para descargar (pestaña Output):")

for path in sorted(OUTPUT.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT.parent)} ({path.stat().st_size / 1e6:.1f} MB)")